# Pokémon LLM Agent with Unsloth on AMD ROCm

**Authors:** Yue Yuan ([yueyuan](https://github.com/yueyuan), yueyuan@amd.com), Bill He ([billishyahao](https://github.com/billishyahao), bill.he@amd.com)

**Phase 13 topic:** *Pokémon LLM Agent with Unsloth.* This notebook is an end-to-end workflow: environment setup, short SFT, inference checks, and a mini-eval using `scripts/eval/showdown_agent_eval.py` / `eval_showdown_agent.py`.

---

## Introduction

This tutorial shows how to fine-tune **Qwen3-4B** with **Unsloth** and **TRL** so the model predicts the next **Pokemon Showdown** battle action (`move …` or `switch …`) from **raw replay logs**. The workflow targets **AMD ROCm** GPUs (e.g., AMD Instinct™ MI300 series or AMD Radeon™ PRO workstation cards) using **bfloat16** training and inference patterns that are stable on ROCm.

You will use a **streaming** Hugging Face dataset so the full replay corpus is never downloaded to disk, which avoids space issues on shared or small volumes. A **50-step** SFT run demonstrates alignment toward strict action formatting; a **mini-eval** reports the same headline metrics as `scripts/eval/eval_showdown_agent.py` for comparison with a **longer-trained reference** on the Hub.

**Published reference weights (optional):** [Hugging Face model repository](https://huggingface.co/GoldenGrapeGentleman1/pokemon-showdown-agent-v6) — same recipe at production scale; this tutorial stays intentionally small.

**Training data (streaming):** [`milkkarten/pokemon-showdown-replays-merged`](https://huggingface.co/datasets/milkkarten/pokemon-showdown-replays-merged) (Apache-2.0)

---

## What you will learn

- Configure **HF cache**, temp directories, and common **ROCm / PyTorch HIP** environment variables from a notebook.
- Install and validate **Unsloth**, **TRL**, **Transformers**, and **Datasets** in a ROCm-capable Python environment.
- Attach **LoRA** adapters and run a **short SFT** job with `SFTTrainer` (`adamw_torch`, no 4-bit load for stability in this tutorial).
- Stream and format replay rows into chat examples without materializing the full dataset.
- Run **inference** and **legality checks** using a synthetic `|request|` block (Showdown-style JSON).
- Run a **mini-evaluation** that shares code with `scripts/eval/showdown_agent_eval.py`.

---

## Prerequisites

### Hardware

- An **AMD GPU** supported by your ROCm stack (Instinct™ datacenter or Radeon™ PRO / consumer, per your container image).
- Sufficient **GPU memory** for **Qwen3-4B** in **bfloat16** with LoRA (typically **≥ 24 GB** VRAM for comfortable headroom; less may work with smaller batches — not covered here).
- **Host disk** or mount with several **gigabytes free** for model weights and checkpoints (exact size depends on cache location).

### Software

- **Linux** environment with **ROCm** and a **PyTorch build that matches your ROCm version** (recommended: an official **AMD ROCm** or partner **Docker** image with JupyterLab).
- **Python3.10+** (this notebook is tested with **3.12** in ROCm containers; adjust if your image differs).
- **Internet access** on first run to download the base model and stream dataset shards (or pre-populate `HF_HOME` offline).

### Accounts and tokens

- A **Hugging Face** account is optional for *reading* public models/datasets; if you hit rate limits, run `huggingface-cli login` or set `HF_TOKEN` per [HF documentation](https://huggingface.co/docs/huggingface_hub/quick-start).
- **Publishing** checkpoints (§8) requires `hf` CLI auth — see [Hugging Face Hub CLI](https://huggingface.co/docs/huggingface_hub/guides/cli).

### Before you start

1. Open this notebook with the **working directory** set to the **repository root** that contains `scripts/eval/showdown_agent_eval.py` (imports in §6–7 depend on it).
2. **Kernel:** select the Python environment where **PyTorch sees your AMD GPU** (`torch.cuda.is_available()` should be true in ROCm builds that expose HIP as CUDA API).
3. Run cells **top to bottom** the first time; if the install cell adds packages, **restart the kernel** and run from the top once.

---

## Tutorial outline

| Step | Section | Purpose |
|------|---------|---------|
| 0 | Environment paths & ROCm env vars | Writable cache, `HF_HOME`, HIP visibility |
| 1 | Install runtime stack | Idempotent pip installs |
| 2 | Validate ROCm runtime | Imports, GPU visibility |
| 3 | Model + LoRA | `Qwen/Qwen3-4B` + PEFT |
| 4 | Data streaming | Replay → chat examples |
| 5 | Short SFT | 50-step alignment demo |
| 6 | Inference + legality | `showdown_agent_eval` helpers |
| 7 | Mini-eval | Same metrics as `eval_showdown_agent.py` |
| 8 | Export / Hub | Optional merge & upload |
| Bonus | GRPO sketch | Optional; requires extra packages |

---

## Design notes (robustness on AMD systems)

1. Dependency installation runs **before** heavy imports.
2. Cache and temp paths **fall back** if `/shared-docker` is unavailable.
3. Data preparation uses **`streaming=True`** end to end — no full 40GB+ local extract.
4. **`load_in_4bit=False`** and **`adamw_torch`** in the tutorial path for fewer ROCm surprises than 4-bit + 8-bit optimizers.
5. Notebook outputs are **cleared** in source so readers see only their own runs.

---

## Post-publication maintenance (for primary authors)

When submitting to the **AMD AI Developer Hub** / `ROCm/gpuaidev-internal` workflow: re-run the full notebook on the **declared ROCm image and GPU SKU** after each major ROCm or dependency bump; update **Prerequisites** if Python, PyTorch, or Unsloth minimums change; keep `showdown_agent_eval.py` and this notebook in sync for metrics definitions. **Primary authors:** Yue Yuan (yueyuan@amd.com), Bill He (bill.he@amd.com).

If you extend training (more steps), document expected VRAM and runtime so reviewers can reproduce.


In [ ]:
# Step 0 — Environment: ROCm/HIP-friendly defaults and writable Hugging Face cache.
# Run this first on the target GPU machine; adjust HIP_VISIBLE_DEVICES if you use multiple GPUs.
import os
import platform
import shutil
from pathlib import Path


def pick_writable_dir(candidates):
    for candidate in candidates:
        path = Path(candidate)
        try:
            path.mkdir(parents=True, exist_ok=True)
            probe = path / ".cursor_write_test"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return path
        except Exception:
            continue
    raise RuntimeError(f"No writable directory found in: {candidates}")


cache_root = pick_writable_dir([
    "/shared-docker/.cache/huggingface",
    "/data/huggingface",
    "/tmp/pokemon-hf-cache",
])
tmp_root = pick_writable_dir([
    "/shared-docker/.cache/tmp",
    "/data/tmp",
    "/tmp/pokemon-tmp",
])

os.environ.setdefault("HIP_VISIBLE_DEVICES", "0")
os.environ.setdefault("ROCR_VISIBLE_DEVICES", os.environ["HIP_VISIBLE_DEVICES"])
os.environ.setdefault("TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL", "1")
os.environ.setdefault("PYTORCH_HIP_ALLOC_CONF", "expandable_segments:False")
os.environ.setdefault("UNSLOTH_SKIP_TORCHVISION_CHECK", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

os.environ["HF_HOME"] = str(cache_root)
os.environ["HF_DATASETS_CACHE"] = str(cache_root / "datasets")
os.environ["TMPDIR"] = str(tmp_root)

Path(os.environ["HF_DATASETS_CACHE"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["TMPDIR"]).mkdir(parents=True, exist_ok=True)

cache_usage = shutil.disk_usage(cache_root)
print(f"platform={platform.platform()}")
print(f"HF_HOME={os.environ['HF_HOME']}")
print(f"HF_DATASETS_CACHE={os.environ['HF_DATASETS_CACHE']}")
print(f"TMPDIR={os.environ['TMPDIR']}")
print(f"HIP_VISIBLE_DEVICES={os.environ['HIP_VISIBLE_DEVICES']}")
print(f"free_space_gb={cache_usage.free / (1024 ** 3):.1f}")

## 1. Install the runtime stack
**Tutorial content — dependencies.** This cell uses only the Python standard library before any ML imports. It detects missing packages, prints versions already installed, and runs `pip install` only for gaps.

On a prebuilt **AMD ROCm** image, keeping the image’s pinned PyTorch + ROCm stack and adding Unsloth on top is usually safer than forcing broad downgrades inside the notebook.

**After installing:** restart the Jupyter kernel once, then **Run All** from the top so import order (Unsloth first) stays correct.

In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

core_runtime = [
    "transformers",
    "trl",
    "datasets",
    "tokenizers",
    "huggingface_hub",
    "accelerate",
    "peft",
    "psutil",
    "sentencepiece",
    "protobuf",
    "tyro",
    "hf_transfer",
    "einops",
]
required_modules = [
    "unsloth",
    "unsloth_zoo",
    "transformers",
    "trl",
    "datasets",
    "tokenizers",
    "huggingface_hub",
    "accelerate",
    "peft",
    "psutil",
    "sentencepiece",
    "google.protobuf",
    "tyro",
    "einops",
]

missing = [name for name in required_modules if importlib.util.find_spec(name) is None]

if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--no-deps",
        "unsloth",
        "unsloth_zoo",
    ])
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        *core_runtime,
    ])
    print("Install completed. Rerun the notebook from the top if this is a fresh environment.")
else:
    print("All required packages are already available.")

print("Detected package versions:")
for package_name in ["unsloth", "transformers", "trl", "datasets", "tokenizers", "huggingface_hub"]:
    try:
        print(f"  {package_name}={importlib.metadata.version(package_name)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {package_name}=missing")

## 2. Validate the AMD ROCm runtime
**Tutorial content — hardware smoke test.** The next cell imports in **Unsloth-first** order, prints package versions, and checks that **PyTorch** reports a GPU (ROCm builds typically expose HIP via the CUDA-like API). If this fails, fix the container or driver stack before training — the AMD AI Developer Hub review expects this step to pass on the declared SKU.

In [ ]:
import unsloth
import datasets
import huggingface_hub
import torch
import transformers
import trl


datasets.config.HF_DATASETS_CACHE = os.environ["HF_DATASETS_CACHE"]

print(f"torch={torch.__version__}")
print(f"transformers={transformers.__version__}")
print(f"trl={trl.__version__}")
print(f"datasets={datasets.__version__}")
print(f"huggingface_hub={huggingface_hub.__version__}")
print(f"unsloth={getattr(unsloth, '__version__', 'unknown')}")
print(f"hip_version={getattr(torch.version, 'hip', None)}")

if not torch.cuda.is_available():
    raise RuntimeError("No ROCm-visible GPU found. This notebook expects an AMD GPU environment.")

print(f"gpu_count={torch.cuda.device_count()}")
print(f"gpu_name={torch.cuda.get_device_name(0)}")
print(f"bf16_supported={torch.cuda.is_bf16_supported()}")

## 3. Load Qwen3-4B and attach LoRA adapters
**Tutorial content — model and adapters.** We load the open **Apache-2.0** base model [`Qwen/Qwen3-4B`](https://huggingface.co/Qwen/Qwen3-4B) and attach **LoRA** so only a small fraction of weights are trained.

For AMD ROCm tutorials we keep **`load_in_4bit=False`** for a stable, reproducible path; you can explore 4-bit later once the baseline passes review hardware. **`attn_implementation="sdpa"`** is set for efficient attention on recent PyTorch builds.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

max_seq_length = 2048
dtype = torch.bfloat16
load_in_4bit = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-4B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    attn_implementation="sdpa",
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"trainable_params={trainable_params / 1e6:.1f}M")
print(f"trainable_ratio={100 * trainable_params / all_params:.2f}%")

## 4. Stream and format replay logs
**Tutorial content — dataset pipeline.** This is the core **raw-log** idea: train directly from **raw** Pokemon Showdown replay logs (not hand-written battle summaries). The code uses **`streaming=True`** end to end and only materializes a **small in-memory subset** (`MAX_TRAIN_SAMPLES`) so you do not need tens of gigabytes of local parquet cache.

In [ ]:
from datasets import Dataset, load_dataset

MIN_RATING = 1400
MAX_TRAIN_SAMPLES = 2000
SHUFFLE_BUFFER = 10_000
MAX_LOG_CHARS = 6000

SYSTEM_TEMPLATE = (
    "You are a Pokemon Showdown battle AI. You play as {side}. "
    "Given the battle log, output your next action. "
    "Format: move <name> OR switch <name>. "
    "Append terastallize if you terastallize this turn."
)


def showdown_fields(line: str) -> list[str]:
    parts = line.strip().split("|")
    while parts and parts[0] == "":
        parts = parts[1:]
    return parts


def extract_winner_side(log_text):
    winner = None
    players = {}
    for line in log_text.split("\n"):
        f = showdown_fields(line)
        if len(f) >= 3 and f[0] == "player":
            players[f[1]] = f[2]
        if len(f) >= 2 and f[0] == "win":
            winner = f[1]
    if not winner:
        return None
    for side, name in players.items():
        if name == winner:
            return side
    return None


def format_sample(example):
    log_text = example["log"]
    side = extract_winner_side(log_text)
    if not side:
        return {"text": ""}

    lines = log_text.strip().split("\n")
    turn_positions = []
    for i, line in enumerate(lines):
        f = showdown_fields(line)
        if len(f) >= 2 and f[0] == "turn":
            try:
                turn_positions.append((int(f[1]), i))
            except ValueError:
                pass

    if len(turn_positions) < 2:
        return {"text": ""}

    target_turn_idx = 0 if len(turn_positions) == 2 else 1
    _, turn_line_idx = turn_positions[target_turn_idx]
    next_turn_line = turn_positions[target_turn_idx + 1][1] if target_turn_idx + 1 < len(turn_positions) else len(lines)

    action = None
    for j in range(turn_line_idx + 1, next_turn_line):
        f = showdown_fields(lines[j])
        if len(f) < 3:
            continue
        if f[0] == "move" and f[1].startswith(f"{side}a:"):
            tera = ""
            start_look = max(0, j - 3)
            end_look = min(len(lines), j + 3)
            if any("terastallize" in lines[k] and side in lines[k] for k in range(start_look, end_look)):
                tera = " terastallize"
            action = f"move {f[2]}{tera}"
            break
        if f[0] == "switch" and f[1].startswith(f"{side}a:"):
            pokemon = f[1].split(": ", 1)[1] if ": " in f[1] else f[1]
            action = f"switch {pokemon}"
            break

    if not action:
        return {"text": ""}

    log_prefix = "\n".join(lines[:turn_line_idx + 1])
    if len(log_prefix) > MAX_LOG_CHARS:
        return {"text": ""}

    messages = [
        {"role": "system", "content": SYSTEM_TEMPLATE.format(side=side)},
        {"role": "user", "content": log_prefix},
        {"role": "assistant", "content": action},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}


print("Streaming replay logs without materializing the full corpus...")
stream = load_dataset(
    "milkkarten/pokemon-showdown-replays-merged",
    split="train",
    streaming=True,
)
stream = stream.shuffle(seed=3407, buffer_size=SHUFFLE_BUFFER)
stream = stream.filter(
    lambda row: "gen9" in str(row.get("formatid") or "").lower().replace(" ", "")
    and (row.get("rating") or 0) >= MIN_RATING
)

train_samples = []
scanned = 0
for row in stream:
    scanned += 1
    formatted = format_sample(row)
    if formatted["text"]:
        train_samples.append(formatted)
    if len(train_samples) >= MAX_TRAIN_SAMPLES:
        break

if not train_samples:
    raise RuntimeError("No training samples were collected. Check dataset access, filters, or disk/network configuration.")

train_dataset = Dataset.from_list(train_samples).shuffle(seed=3407)
print(f"collected_examples={len(train_dataset)}")
print(f"replays_scanned={scanned}")
print(train_dataset[0]["text"][:1000])

## 5. Quick validation SFT
**Tutorial content — training.** This step is **not** meant to maximize ladder Elo; it is a **50-step** demonstration that the model shifts from chatty text toward **short `move` / `switch` commands**. For AMD Hub review, confirm this cell completes on your target GPU without OOM; increase `max_steps` only after you document VRAM and runtime.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

output_dir = "outputs/pokemon_showdown_agent_tutorial"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=50,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_torch",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=output_dir,
        save_steps=50,
        save_total_limit=2,
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)

## 6. Inference sanity check
**Tutorial content — qualitative check.** A healthy post-SFT output should be compact and action-shaped. After only 50 steps it will not be perfect, but it should usually look like `move ...` or `switch ...` instead of free-form commentary.

This cell checks more than a single regex on the first token:

- **Line shape:** one non-empty line, matching `move ...` or `switch ...`, with an optional trailing `terastallize`.
- **Log consistency (lightweight):** we infer p2’s bench only from `|switch|p2*` lines already present in the pasted log (not from unrevealed team preview). A `switch` target must match one of those species and cannot be the current p2a. If the log has not yet shown every teammate, this check can false-negative a legal switch.
- **Turn legality (`|request|` JSON):** when the log contains a Showdown `|request|{...}` line for the current decision, we treat it as authoritative for this turn: `move` must match a non-disabled entry in `active[].moves`, `switch` must match a non-fainted, non-active Pokémon in `side.pokemon`, and `terastallize` is only allowed if `canTerastallize` is set. Many public replays omit `|request|`; paste it from a live battle or bot client when you need true legality checks.

This still does **not** cover doubles targeting, some forced-choice edge cases, or server-side PP tracking beyond what the request encodes. Treat these checks as **necessary but not sufficient** for a production ladder agent.

Code: `scripts/eval/showdown_agent_eval.py` (`tutorial_demo_log_with_request`, `validate_action_against_log`).

In [ ]:
import sys
from pathlib import Path

import torch

_repo = Path.cwd().resolve()
for base in [_repo, *_repo.parents]:
    if (base / "scripts" / "eval" / "showdown_agent_eval.py").is_file():
        sys.path.insert(0, str(base / "scripts" / "eval"))
        break
else:
    raise FileNotFoundError(
        "Could not find scripts/eval/showdown_agent_eval.py — set cwd to the pokemon repo root."
    )

from showdown_agent_eval import (
    postprocess_agent_response,
    tutorial_demo_log_with_request,
    validate_action_against_log,
)

model.eval()
FastLanguageModel.for_inference(model)

sys_msg = (
    "You are a Pokemon Showdown battle AI. You play as p2. "
    "Given the battle log, output your next action. "
    "Format: move <name> OR switch <name>. "
    "Append terastallize if you terastallize this turn."
)
user_msg = tutorial_demo_log_with_request()

messages = [
    {"role": "system", "content": sys_msg},
    {"role": "user", "content": user_msg},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=False,
    )

full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
if "<|im_start|>assistant\n" in full_response:
    response = full_response.split("<|im_start|>assistant\n")[-1]
else:
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
response = postprocess_agent_response(response)

first_line = response.splitlines()[0].strip() if response.strip() else ""
checks = validate_action_against_log(first_line, user_msg)

print("--- AI AGENT PREDICTION ---")
print(response)
print("--- legality & format (showdown_agent_eval) ---")
print(f"structure_ok={checks['structure_ok']}")
print(f"single_line_ok={checks['single_line_ok']}")
print(f"request_present={checks['request_present']}")
if checks["move_name_nonempty"] is not None:
    print(f"move_name_nonempty={checks['move_name_nonempty']}")
if checks["switch_target_ok"] is not None:
    print(f"switch_target_ok={checks['switch_target_ok']}")
if checks["move_legal_in_request"] is not None:
    print(f"move_legal_in_request={checks['move_legal_in_request']}")
if checks["switch_legal_in_request"] is not None:
    print(f"switch_legal_in_request={checks['switch_legal_in_request']}")
if checks.get("parsed") and checks["parsed"].get("tera") and checks["tera_legal_in_request"] is not None:
    print(f"tera_legal_in_request={checks['tera_legal_in_request']}")
print(f"notes={checks['notes']}")


## 7. Mini-eval (reference protocol)

**Tutorial content — quantitative check.** After the short SFT run, we report **the same headline metrics** as `scripts/eval/eval_showdown_agent.py`, implemented in `scripts/eval/showdown_agent_eval.py`:

- **Valid format:** output contains a parsable `move …` or `switch …` (same regex as the standalone script).
- **Type match:** predicted move vs switch matches the ground-truth action type.
- **Exact match:** normalized target string matches the replay’s winner action for that turn.

This notebook uses a **small** `EVAL_SAMPLES` (streaming, held-out from training) so the loop stays fast on tutorial hardware. A fully trained reference run typically uses larger `samples` (50–200+). Expect **low exact match** after only 50 optimizer steps; the point is to compare checkpoints and to align with `eval_showdown_agent.py`, not to claim ladder strength.

Results are written next to your checkpoint under `eval_tutorial_protocol.json`.


In [ ]:
import random
import sys
from pathlib import Path

import torch
from datasets import load_dataset

_repo = Path.cwd().resolve()
for base in [_repo, *_repo.parents]:
    if (base / "scripts" / "eval" / "showdown_agent_eval.py").is_file():
        sys.path.insert(0, str(base / "scripts" / "eval"))
        REPO_ROOT = base
        break
else:
    raise FileNotFoundError(
        "Could not find scripts/eval/showdown_agent_eval.py. "
        "Open this notebook with the working directory set to the pokemon repo root."
    )

from showdown_agent_eval import (
    build_test_samples,
    eval_showdown_agent_batch,
    print_metrics_summary,
    save_metrics_json,
)

EVAL_SEED = 3407
EVAL_SAMPLES = 24
random.seed(EVAL_SEED)
torch.manual_seed(EVAL_SEED)

print("Streaming held-out eval examples (same sampling logic as eval_showdown_agent.py)...")
stream = load_dataset(
    "milkkarten/pokemon-showdown-replays-merged",
    split="train",
    streaming=True,
)
stream = stream.shuffle(seed=EVAL_SEED + 11, buffer_size=10_000)
eval_samples = build_test_samples(
    stream, EVAL_SAMPLES, seed=EVAL_SEED, max_scans=400_000
)
print(f"collected_eval_samples={len(eval_samples)}")

if not eval_samples:
    stream2 = load_dataset(
        "milkkarten/pokemon-showdown-replays-merged",
        split="train",
        streaming=True,
    ).shuffle(seed=EVAL_SEED + 99, buffer_size=10_000)
    eval_samples = build_test_samples(
        stream2,
        EVAL_SAMPLES,
        seed=EVAL_SEED,
        require_gen9=False,
        max_scans=400_000,
    )
    print(f"collected_eval_samples_relaxed_gen={len(eval_samples)}")

if not eval_samples:
    raise RuntimeError("No eval samples collected; check dataset access or filters.")

model.eval()
metrics, _rows = eval_showdown_agent_batch(
    model,
    tokenizer,
    eval_samples,
    max_new_tokens=30,
    use_cache=False,
)

print_metrics_summary(metrics, title="Tutorial SFT checkpoint — mini-eval (reference protocol)")
out_path = REPO_ROOT / Path(output_dir) / "eval_tutorial_protocol.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
save_metrics_json(
    str(out_path),
    metrics,
    extra={
        "protocol": "showdown_agent_eval.eval_showdown_agent_batch",
        "eval_samples": EVAL_SAMPLES,
        "eval_seed": EVAL_SEED,
        "output_dir": output_dir,
        "trainer_max_steps": 50,
        "note": "Compare to a full checkpoint using the same metrics; increase EVAL_SAMPLES or train longer for tighter numbers.",
    },
)
print(f"Wrote metrics to {out_path}")


## 8. Export and publish
**Tutorial content — artifacts.** If you want a **standalone merged** checkpoint for sharing or for Hugging Face Hub upload, export under the same `outputs/` tree as the trainer.

For **AMD AI Developer Hub** intake (`ROCm/gpuaidev-internal`), follow the project’s **branch + PR** workflow: add this `.ipynb` in the folder maintainers specify, update the repo **README** and **Sphinx** TOC as required, and note the **ROCm image + GPU SKU** you used for testing.

```python
export_dir = f"{output_dir}/merged_model"
model.save_pretrained_merged(export_dir, tokenizer, save_method="merged_16bit")
```

Suggested Hugging Face commands for a tutorial-only release:

```bash
hf auth whoami
hf repos create your-username/pokemon-showdown-agent-tutorial-demo --type model --exist-ok
hf upload-large-folder your-username/pokemon-showdown-agent-tutorial-demo outputs/pokemon_showdown_agent_tutorial/merged_model
hf upload your-username/pokemon-showdown-agent-tutorial-demo pokemon_llm_agent_unsloth_rocm_tutorial.ipynb pokemon_llm_agent_unsloth_rocm_tutorial.ipynb
```

The published reference weights remain on Hugging Face at the link in the Introduction.

## Bonus: GRPO sketch for the next stage
**Optional extension (not required for Hub smoke tests).** After SFT, you can experiment with **GRPO** to reward formatting or richer objectives. Many `trl` builds import **`mergekit`** and **`llm_blender`** when loading `GRPOTrainer`.

To keep the main SFT environment stable, the next cell does **not** auto-install those packages. Instead, it checks whether the optional GRPO stack is available and cleanly skips the setup with an actionable message if it is not.

Run this section only after the earlier SFT/data cells have completed, because it reuses `train_samples`, `Dataset`, and `is_bfloat16_supported()` from the main notebook flow.

In [ ]:
import importlib.util

grpo_trainer = None
missing_optional = [
    package_name
    for module_name, package_name in [("mergekit", "mergekit"), ("llm_blender", "llm-blender")]
    if importlib.util.find_spec(module_name) is None
]

if missing_optional:
    print("Skipping Bonus/GRPO setup because optional packages are missing.")
    print("Install them in a separate or disposable environment if you want this section:")
    print(f"  pip install {' '.join(missing_optional)}")
else:
    try:
        from trl import GRPOConfig, GRPOTrainer
    except Exception as exc:
        print("Skipping Bonus/GRPO setup because TRL could not import GRPOTrainer cleanly.")
        print(repr(exc))
    else:
        def format_reward_func(prompts, completions, **kwargs):
            rewards = []
            for completion in completions:
                text = completion[0]["content"] if isinstance(completion, list) else str(completion)
                cmd_match = re.search(r"^(move\s+.+|switch\s+.+)$", text.strip(), re.IGNORECASE)
                rewards.append(5.0 if cmd_match else -3.0)
            return rewards

        grpo_prompts = [
            {"prompt": sample["text"].split("<|im_start|>assistant")[0] + "<|im_start|>assistant\n"}
            for sample in train_samples[:50]
        ]
        grpo_dataset = Dataset.from_list(grpo_prompts)

        grpo_config = GRPOConfig(
            output_dir="outputs/pokemon_showdown_agent_grpo_demo",
            learning_rate=3e-6,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            num_generations=4,
            max_completion_length=128,
            temperature=1.3,
            max_steps=10,
            logging_steps=1,
            report_to="none",
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
        )

        grpo_trainer = GRPOTrainer(
            model=model,
            reward_funcs=[format_reward_func],
            args=grpo_config,
            train_dataset=grpo_dataset,
        )
        print("GRPO setup is ready. Uncomment `grpo_trainer.train()` when desired.")

# Uncomment when you want to test the RL extension path.
# if grpo_trainer is not None:
#     grpo_trainer.train()